# CLT Cascade Classifier — Full Training Run

Trains BERT-base and RoBERTa-base (Stage 1 salience + Stage 2 CLT dimensions) on GPU.
Outputs are saved to Google Drive after each run so a Colab disconnect loses nothing.

**Run cells in order on first use. Each cell is independently re-runnable after a reconnect.**

| Cell | What it does |
|------|--------------|
| 1 | GPU check — hard-stops if no GPU |
| 2 | Mount Google Drive |
| 3 | Clone repo + install dependencies |
| 4 | Upload data files + verify counts |
| 5 | BERT Stage 1 training |
| 6 | BERT Stage 2 training |
| 7 | RoBERTa Stage 1 training |
| 8 | RoBERTa Stage 2 training |
| 9 | Results summary table |
| 10 | Commit RoBERTa configs to GitHub |

## Cell 1 — GPU Check

Verifies a GPU is attached. Hard-stops if not — training on CPU is not viable.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("No GPU detected.")
    print("Switch to GPU runtime: Runtime > Change runtime type > T4 GPU, then re-run all cells.")
    raise RuntimeError("GPU required. Switch runtime and reconnect.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU detected: {gpu_name}  ({vram_gb:.1f} GB)")
print(f"torch version: {torch.__version__}")
print("Cell 1 OK.")

## Cell 2 — Mount Google Drive

Creates `/content/drive/MyDrive/CLT_Thesis/outputs/` for persistent checkpoint storage.
Re-running this cell after a reconnect is safe.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=False)

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

print(f"Drive output folder: {DRIVE_OUTPUT_DIR}")
print("Contents:", os.listdir(DRIVE_OUTPUT_DIR) or "(empty)")
print("Cell 2 OK.")

## Cell 3 — Clone Repo + Install Dependencies

- Sets `HF_HOME` so model weights are cached under `/content/hf_cache/` (not the default)
- Clones the repo if it does not already exist (safe to re-run after reconnect)
- Installs pinned dependencies from `requirements.txt`

In [ ]:
import os
os.environ["HF_HOME"] = "/content/hf_cache"

# Clone repo
!git clone https://github.com/KrisHoffmann/Bachelor-Thesis-IK.git /content/Bachelor-Thesis-IK
%cd /content/Bachelor-Thesis-IK

# Install only what Colab does not already have
# torch, numpy, sklearn are pre-installed — do NOT reinstall them
!pip install -q "transformers>=4.41.0" accelerate datasets krippendorff pyyaml

print("\nRepo contents:")
!ls -la
print("\nReady to proceed to Cell 4.")

## Cell 4 — Upload Data Files

Upload `train.json`, `dev.json`, and `test.json` when prompted.
The cell will verify counts (723 / 176 / 155) and hard-stop on any mismatch.

**Do not skip this cell.** Re-runnable after reconnect — files already in `data/processed/` are detected and the upload step is skipped.

In [ ]:
import os
import shutil
import sys

sys.path.insert(0, "/content/Bachelor-Thesis-IK")
os.makedirs("data/processed", exist_ok=True)

REQUIRED = ["train.json", "dev.json", "test.json"]
already_present = all(os.path.exists(f"data/processed/{f}") for f in REQUIRED)

if already_present:
    print("Data files already present in data/processed/ — skipping upload.")
else:
    print("Upload train.json, dev.json, test.json now — do this before continuing")
    print("(Use the file chooser that appears below)")
    from google.colab import files
    uploaded = files.upload()  # blocks until user selects files

    if not uploaded:
        raise RuntimeError("No files uploaded. Re-run this cell and upload all three JSON files.")

    for fname, content in uploaded.items():
        dest = f"data/processed/{fname}"
        # files.upload() writes to CWD; move to data/processed/
        if os.path.exists(fname) and not os.path.exists(dest):
            shutil.move(fname, dest)
        elif os.path.exists(fname):
            shutil.move(fname, dest)
        print(f"  Saved: {dest}")

    missing = [f for f in REQUIRED if not os.path.exists(f"data/processed/{f}")]
    if missing:
        raise RuntimeError(
            f"Missing files after upload: {missing}\n"
            "Re-run this cell and upload all three JSON files."
        )

# Verify counts — hard-fail on mismatch
from src.data import verify_splits
try:
    verify_splits("data/processed")
except AssertionError as e:
    raise RuntimeError(f"Data verification failed: {e}") from e

print("Data verified. Ready to train.")

## Cell 5 — BERT-base Stage 1 Training

Fine-tunes `bert-base-uncased` for salience classification (binary, class-weighted).
Config: `configs/bert_stage1.yaml` — 5 epochs, batch 16, lr 2e-5.
Outputs saved to `outputs/bert_stage1/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "bert_stage1"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
METRICS_PATH = f"{LOCAL_OUT}/metrics.json"

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  BERT Stage 1 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", "configs/bert_stage1.yaml", "--stage", "1"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best dev macro-F1
with open(METRICS_PATH) as f:
    m = json.load(f)
print(f"\nBERT Stage 1 — Dev macro-F1: {m['macro_f1']:.4f}  |  Accuracy: {m['accuracy']:.4f}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"BERT Stage 1 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 6 — BERT-base Stage 2 Training

Trains four parallel 3-class CLT heads (Temporal / Spatial / Social / Hypothetical)
on the 621 salient training sentences.
Config: `configs/bert_stage2.yaml` — 5 epochs, batch 16, lr 2e-5, warmup 10%.
Outputs saved to `outputs/bert_stage2/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "bert_stage2"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
METRICS_PATH = f"{LOCAL_OUT}/epoch_metrics.json"

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  BERT Stage 2 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", "configs/bert_stage2.yaml", "--stage", "2"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best epoch metrics (highest mean_macro_f1)
with open(METRICS_PATH) as f:
    epoch_records = json.load(f)

best = max(epoch_records, key=lambda r: r["mean_macro_f1"])
print(f"\nBERT Stage 2 — Best epoch {best['epoch']}  |  Mean macro-F1: {best['mean_macro_f1']:.4f}")
for dim in ("temporal", "spatial", "social", "hypothetical"):
    f1 = best[dim]["macro_f1"]
    collapsed = best[dim].get("collapsed", False)
    note = "  [COLLAPSE]" if collapsed else ""
    print(f"  {dim:<14}: macro-F1 = {f1:.4f}{note}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"BERT Stage 2 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 7 — RoBERTa-base Stage 1 Training

Creates `configs/roberta_stage1.yaml` (copy of bert_stage1.yaml with `roberta-base`)
and runs the same Stage 1 pipeline. The training script is reused unchanged.
Outputs saved to `outputs/roberta_stage1/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

import yaml

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "roberta_stage1"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
ROBERTA_CFG = "configs/roberta_stage1.yaml"
METRICS_PATH = f"{LOCAL_OUT}/metrics.json"

# Create RoBERTa Stage 1 config (derived from bert_stage1.yaml)
with open("configs/bert_stage1.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["model_name"] = "roberta-base"
cfg["output_dir"] = f"outputs/{RUN_NAME}"
cfg["run_name"] = RUN_NAME
with open(ROBERTA_CFG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Created {ROBERTA_CFG}")

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  RoBERTa Stage 1 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", ROBERTA_CFG, "--stage", "1"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best dev macro-F1
with open(METRICS_PATH) as f:
    m = json.load(f)
print(f"\nRoBERTa Stage 1 — Dev macro-F1: {m['macro_f1']:.4f}  |  Accuracy: {m['accuracy']:.4f}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"RoBERTa Stage 1 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 8 — RoBERTa-base Stage 2 Training

Creates `configs/roberta_stage2.yaml` (copy of bert_stage2.yaml with `roberta-base`)
and runs the same Stage 2 pipeline.
Outputs saved to `outputs/roberta_stage2/` and immediately backed up to Drive.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

import yaml

os.environ["HF_HOME"] = "/content/hf_cache"
sys.path.insert(0, "/content/Bachelor-Thesis-IK")

DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/CLT_Thesis/outputs"
RUN_NAME = "roberta_stage2"
LOCAL_OUT = f"outputs/{RUN_NAME}"
DRIVE_OUT = f"{DRIVE_OUTPUT_DIR}/{RUN_NAME}"
ROBERTA_CFG = "configs/roberta_stage2.yaml"
METRICS_PATH = f"{LOCAL_OUT}/epoch_metrics.json"

# Create RoBERTa Stage 2 config (derived from bert_stage2.yaml)
with open("configs/bert_stage2.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["model_name"] = "roberta-base"
cfg["output_dir"] = f"outputs/{RUN_NAME}"
cfg["run_name"] = RUN_NAME
with open(ROBERTA_CFG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)
print(f"Created {ROBERTA_CFG}")

t_start = datetime.now()
print(f"[{t_start:%Y-%m-%d %H:%M:%S}]  RoBERTa Stage 2 training started")

try:
    result = subprocess.run(
        [sys.executable, "src/train.py", "--config", ROBERTA_CFG, "--stage", "2"],
        check=True
    )
except subprocess.CalledProcessError as e:
    print(f"\nTraining failed with exit code {e.returncode}.")
    raise

t_end = datetime.now()
elapsed = t_end - t_start
print(f"[{t_end:%Y-%m-%d %H:%M:%S}]  Training finished  (elapsed: {elapsed})")

# Load and print best epoch metrics
with open(METRICS_PATH) as f:
    epoch_records = json.load(f)

best = max(epoch_records, key=lambda r: r["mean_macro_f1"])
print(f"\nRoBERTa Stage 2 — Best epoch {best['epoch']}  |  Mean macro-F1: {best['mean_macro_f1']:.4f}")
for dim in ("temporal", "spatial", "social", "hypothetical"):
    f1 = best[dim]["macro_f1"]
    collapsed = best[dim].get("collapsed", False)
    note = "  [COLLAPSE]" if collapsed else ""
    print(f"  {dim:<14}: macro-F1 = {f1:.4f}{note}")

# Copy to Drive immediately
if os.path.isdir(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUT, DRIVE_OUT)

n_files = sum(len(files) for _, _, files in os.walk(DRIVE_OUT))
total_bytes = sum(
    os.path.getsize(os.path.join(root, f))
    for root, _, files in os.walk(DRIVE_OUT) for f in files
)
print(f"RoBERTa Stage 2 saved to Drive: {n_files} files, {total_bytes/1e6:.1f} MB")
print(f"  Path: {DRIVE_OUT}")

## Cell 9 — Results Summary Table

Loads metrics from all four completed runs and prints a comparison table.
Missing runs are skipped with a warning rather than crashing.
This table goes directly into the thesis.

In [ ]:
import json
import os

DIMS = ("temporal", "spatial", "social", "hypothetical")

def load_stage1_metrics(run_name):
    path = f"outputs/{run_name}/metrics.json"
    if not os.path.exists(path):
        # Also check Drive as fallback
        path = f"/content/drive/MyDrive/CLT_Thesis/outputs/{run_name}/metrics.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)

def load_stage2_metrics(run_name):
    path = f"outputs/{run_name}/epoch_metrics.json"
    if not os.path.exists(path):
        path = f"/content/drive/MyDrive/CLT_Thesis/outputs/{run_name}/epoch_metrics.json"
    if not os.path.exists(path):
        return None
    with open(path) as f:
        records = json.load(f)
    return max(records, key=lambda r: r["mean_macro_f1"])

def fmt(val):
    return f"{val:.3f}" if val is not None else "  N/A "

rows = []
for model_label, s1_run, s2_run in [
    ("BERT-base",  "bert_stage1",    "bert_stage2"),
    ("RoBERTa-base", "roberta_stage1", "roberta_stage2"),
]:
    s1 = load_stage1_metrics(s1_run)
    s2 = load_stage2_metrics(s2_run)

    if s1 is None:
        print(f"WARNING: Stage 1 metrics not found for {model_label} ({s1_run}) — run skipped.")
    if s2 is None:
        print(f"WARNING: Stage 2 metrics not found for {model_label} ({s2_run}) — run skipped.")

    s1_f1   = s1["macro_f1"] if s1 else None
    s2_temp = s2["temporal"]["macro_f1"] if s2 else None
    s2_spat = s2["spatial"]["macro_f1"] if s2 else None
    s2_soc  = s2["social"]["macro_f1"] if s2 else None
    s2_hypo = s2["hypothetical"]["macro_f1"] if s2 else None
    s2_mean = s2["mean_macro_f1"] if s2 else None

    rows.append((model_label, s1_f1, s2_temp, s2_spat, s2_soc, s2_hypo, s2_mean))

# Print table
header = f"{'Model':<14} | {'S1 macro-F1':>11} | {'S2 Temporal':>11} | {'S2 Spatial':>10} | {'S2 Social':>9} | {'S2 Hypothetical':>15} | {'S2 Mean':>7}"
sep = "-" * len(header)
print(sep)
print(header)
print(sep)
for model_label, s1_f1, s2_temp, s2_spat, s2_soc, s2_hypo, s2_mean in rows:
    print(
        f"{model_label:<14} | {fmt(s1_f1):>11} | {fmt(s2_temp):>11} | {fmt(s2_spat):>10} | "
        f"{fmt(s2_soc):>9} | {fmt(s2_hypo):>15} | {fmt(s2_mean):>7}"
    )
print(sep)
print("\nNote: Stage 2 Temporal and Spatial macro-F1 may be low due to ~90% N/A class imbalance.")
print("      COLLAPSE means the model predicted only one class for that dimension.")

## Cell 10 — Commit RoBERTa Configs to GitHub

Commits `configs/roberta_stage1.yaml` and `configs/roberta_stage2.yaml` to the repo.
If the push fails with an authentication error, step-by-step PAT instructions are printed.

In [ ]:
import subprocess

def run_git(args, check=True):
    result = subprocess.run(["git"] + args, capture_output=True, text=True)
    return result

# Confirm the two config files exist before committing
import os
for cfg_file in ["configs/roberta_stage1.yaml", "configs/roberta_stage2.yaml"]:
    if not os.path.exists(cfg_file):
        raise FileNotFoundError(
            f"{cfg_file} not found — run Cells 7 and 8 first to generate the RoBERTa configs."
        )

# Set git identity (Colab has no global git config by default)
run_git(["config", "user.email", "colab-training@thesis"])
run_git(["config", "user.name", "Colab Training Run"])

# Stage the two new config files only
r = run_git(["add", "configs/roberta_stage1.yaml", "configs/roberta_stage2.yaml"])
if r.returncode != 0:
    raise RuntimeError(f"git add failed: {r.stderr}")

# Check if there is actually something to commit
r = run_git(["diff", "--cached", "--name-only"])
staged = r.stdout.strip()
if not staged:
    print("Nothing new to commit — RoBERTa configs already committed.")
else:
    print(f"Staged files:\n{staged}")
    r = run_git(["commit", "-m", "feat: add RoBERTa configs from Colab training run"])
    if r.returncode != 0:
        raise RuntimeError(f"git commit failed: {r.stderr}")
    print("Committed.")

# Push
r = run_git(["push", "origin", "main"])
if r.returncode == 0:
    print("Pushed to origin/main successfully.")
else:
    stderr = r.stderr
    if any(word in stderr.lower() for word in ["authentication", "auth", "403", "username", "password", "token"]):
        print("Push failed: GitHub authentication required.")
        print()
        print("To push from Colab, set up a Personal Access Token (PAT):")
        print("  1. Go to https://github.com/settings/tokens")
        print("  2. Generate new token (classic) — tick the 'repo' scope.")
        print("  3. Copy the token (shown only once).")
        print("  4. In a new Colab cell, run:")
        print("       from google.colab import userdata")
        print("       import subprocess")
        print("       token = userdata.get('GITHUB_TOKEN')  # or paste directly")
        print("       subprocess.run(['git', 'remote', 'set-url', 'origin',")
        print("           f'https://{token}@github.com/KrisHoffmann/Bachelor-Thesis-IK.git'])")
        print("       subprocess.run(['git', 'push', 'origin', 'main'])")
        print()
        print("Commit is saved locally — nothing is lost. Push when auth is configured.")
    else:
        print(f"Push failed with unexpected error:\n{stderr}")
        print("Commit is saved locally. Investigate the error above before retrying.")